**문제해결** 

diamonds.csv 데이터셋을 이용해서 가격 예측 모델 학습을 진행하고 평가 결과를 확인

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn import set_config

set_config(display="text") 

In [3]:
route = "C:\\Users\\EZ\\Desktop\\루키즈\\Python\\Python ML,DL\\260730\\diamonds.csv"

In [4]:
data = pd.read_csv(route, encoding="cp949", low_memory=False)

In [5]:
df = pd.DataFrame(data)

df = df.drop(columns=["Unnamed: 0"])

df

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75
...,...,...,...,...,...,...,...,...,...,...
53935,0.72,Ideal,D,SI1,60.8,57.0,2757,5.75,5.76,3.50
53936,0.72,Good,D,SI1,63.1,55.0,2757,5.69,5.75,3.61
53937,0.70,Very Good,D,SI1,62.8,60.0,2757,5.66,5.68,3.56
53938,0.86,Premium,H,SI2,61.0,58.0,2757,6.15,6.12,3.74


In [6]:
columns = [
    "cut",
    "color",
    "clarity"
]

for column in columns:
    encoder = LabelEncoder()
    df[column] = encoder.fit_transform(df[column])

df

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,2,1,3,61.5,55.0,326,3.95,3.98,2.43
1,0.21,3,1,2,59.8,61.0,326,3.89,3.84,2.31
2,0.23,1,1,4,56.9,65.0,327,4.05,4.07,2.31
3,0.29,3,5,5,62.4,58.0,334,4.20,4.23,2.63
4,0.31,1,6,3,63.3,58.0,335,4.34,4.35,2.75
...,...,...,...,...,...,...,...,...,...,...
53935,0.72,2,0,2,60.8,57.0,2757,5.75,5.76,3.50
53936,0.72,1,0,2,63.1,55.0,2757,5.69,5.75,3.61
53937,0.70,4,0,2,62.8,60.0,2757,5.66,5.68,3.56
53938,0.86,3,4,3,61.0,58.0,2757,6.15,6.12,3.74


In [7]:
x = df.drop(columns=["price"])
y = df["price"]

In [8]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42
)

In [9]:
model = LinearRegression()

model.fit(x_train, y_train)

LinearRegression()

In [10]:
y_pred = model.predict(x_test)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test,y_pred)

In [11]:
print("mse : ", mse)
print("rmse : ", rmse)
print("mae : ", mae)
print("r2 : ",r2)

mse :  1825912.9915253515
rmse :  1351.2634796831267
mae :  858.7084697710105
r2 :  0.8851397433679629


In [12]:
print("범주형 변수 수동 원-핫 인코딩")

# 다중공선성 방지를 위해 drop_first=True 설정
# 문자열 컬럼들이 0과 1의 수치형 컬럼들로 분리됩니다.
df_encoded = pd.get_dummies(df, columns=['cut', 'color', 'clarity'], drop_first=True, dtype=int)

# 피처(X)와 타겟(y) 분리
X = df_encoded.drop(columns=['price'])
y = df_encoded['price']

# 학습 및 검증 데이터 분할 (8:2)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 원본 데이터프레임 백업 (나중에 SHAP 변환용 데이터로 활용)
X_train_raw = X_train.copy()
X_test_raw = X_test.copy()

범주형 변수 수동 원-핫 인코딩


In [13]:
print("수치형 변수 수동 스케일링")
numeric_features = ['carat', 'depth', 'table', 'x', 'y', 'z']

scaler = StandardScaler()

# 훈련 데이터의 수치형 컬럼들만 표준화 기준 학습 및 변환
X_train[numeric_features] = scaler.fit_transform(X_train[numeric_features])

# 테스트 데이터는 훈련 데이터 기준으로 변환만 수행
X_test[numeric_features] = scaler.transform(X_test[numeric_features])

수치형 변수 수동 스케일링


NameError: name 'StandardScaler' is not defined

In [ ]:
print("\n[모델 1] 기본 선형 회귀 학습")
start_time = time.time()

lr_basic = LinearRegression()
lr_basic.fit(X_train, y_train)
y_pred_basic = lr_basic.predict(X_test)

time_basic = time.time() - start_time

rmse_basic = np.sqrt(mean_squared_error(y_test, y_pred_basic))
r2_basic = r2_score(y_test, y_pred_basic)

In [ ]:
print("수동 다항 피처 생성")
start_time = time.time()

# degree=2 설정으로 제곱항 및 교차항 생성 변환기 정의
poly = PolynomialFeatures(degree=2, include_bias=False)

# 전처리가 완료된 X_train과 X_test 행렬 전체를 다항식으로 확장합니다.
X_train_poly = poly.fit_transform(X_train_raw)
X_test_poly = poly.transform(X_test_raw)

# 변환된 다항 데이터를 가지고 새 선형 회귀 모델 학습
lr_poly = LinearRegression()
lr_poly.fit(X_train_poly, y_train)
y_pred_poly = lr_poly.predict(X_test_poly)

time_poly = time.time() - start_time
print(time_poly)

rmse_poly = np.sqrt(mean_squared_error(y_test, y_pred_poly))
r2_poly = r2_score(y_test, y_pred_poly)